PySpark DataFrame Case Study: Employee Performance Review Analysis
Objective

The objective of this case study is to analyze employee performance review data using PySpark DataFrame operations such as filtering, handling null values, grouping, aggregation, joins, unions, SQL queries, and window functions.

SparkSession is used to create the Spark application.
col helps reference DataFrame columns.
avg calculates averages.
coalesce replaces null values.
Window is used for window functions.

In [0]:
#Import Required Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, coalesce
from pyspark.sql.window import Window

In [0]:
#Initialize Spark Session
spark = SparkSession.builder \
    .appName("Employee Performance Review Analysis") \
    .getOrCreate()

#This creates a Spark session named Employee Performance Review Analysis.

In [0]:
#Create Sample Data
data = [
    (1, '2024-01-10', 'Engineering', 5, 'John'),
    (2, '2024-01-11', 'HR', 4, 'Jane'),
    (3, '2024-01-12', 'Sales', 3, 'Sam'),
    (4, '2024-02-01', 'Engineering', 5, 'John'),
    (1, '2024-03-10', 'Engineering', 4, 'Jane'),
    (2, '2024-03-11', 'HR', None, 'Sam')
]

columns = ["emp_id", "review_date", "department", "rating", "reviewer"]

df = spark.createDataFrame(data, schema=columns)

df.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
#Filter Operation
df_filtered = df.filter(df.rating >= 4)

df_filtered.show()

#The filter() function selects rows where the rating is greater than or equal to 4.

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
+------+-----------+-----------+------+--------+



In [0]:
#Handle Null Values
window_spec = Window.partitionBy('department')

df_filled = df.withColumn(
    'rating',
    coalesce(df.rating, avg('rating').over(window_spec))
)

df_filled.show()

#partitionBy('department') creates groups by department.
#avg('rating').over(window_spec) calculates department average.
#coalesce() replaces null ratings with the calculated average.

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|   5.0|    John|
|     4| 2024-02-01|Engineering|   5.0|    John|
|     1| 2024-03-10|Engineering|   4.0|    Jane|
|     2| 2024-01-11|         HR|   4.0|    Jane|
|     2| 2024-03-11|         HR|   4.0|     Sam|
|     3| 2024-01-12|      Sales|   3.0|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
#Drop Duplicates
df_no_duplicates = df.dropDuplicates(['emp_id', 'review_date'])

df_no_duplicates.show()
#dropDuplicates() removes duplicate rows based on selected columns.

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
#Select Specific Columns
df_selected = df.select('emp_id', 'department', 'rating')

df_selected.show()

+------+-----------+------+
|emp_id| department|rating|
+------+-----------+------+
|     1|Engineering|     5|
|     2|         HR|     4|
|     3|      Sales|     3|
|     4|Engineering|     5|
|     1|Engineering|     4|
|     2|         HR|  NULL|
+------+-----------+------+



In [0]:
#Grouping and Aggregation
df_grouped = df.groupBy('department').agg(avg('rating').alias('avg_rating'))

df_grouped.show()

+-----------+-----------------+
| department|       avg_rating|
+-----------+-----------------+
|Engineering|4.666666666666667|
|         HR|              4.0|
|      Sales|              3.0|
+-----------+-----------------+



In [0]:
#Joining DataFrames
employee_data = [
    (1, 'Alice'),
    (2, 'Bob'),
    (3, 'Charlie'),
    (4, 'David')
]

employee_columns = ['emp_id', 'employee_name']

df_employees = spark.createDataFrame(employee_data, employee_columns)

df_joined = df.join(df_employees, on='emp_id', how='inner')

df_joined.show()

+------+-----------+-----------+------+--------+-------------+
|emp_id|review_date| department|rating|reviewer|employee_name|
+------+-----------+-----------+------+--------+-------------+
|     1| 2024-01-10|Engineering|     5|    John|        Alice|
|     2| 2024-01-11|         HR|     4|    Jane|          Bob|
|     3| 2024-01-12|      Sales|     3|     Sam|      Charlie|
|     4| 2024-02-01|Engineering|     5|    John|        David|
|     1| 2024-03-10|Engineering|     4|    Jane|        Alice|
|     2| 2024-03-11|         HR|  NULL|     Sam|          Bob|
+------+-----------+-----------+------+--------+-------------+



In [0]:
#Union of DataFrames
new_reviews = [
    (5, '2024-04-01', 'Finance', 5, 'Lisa')
]

df_new_reviews = spark.createDataFrame(new_reviews, columns)

df_union = df.union(df_new_reviews)

df_union.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
|     5| 2024-04-01|    Finance|     5|    Lisa|
+------+-----------+-----------+------+--------+



In [0]:
#Temporary View and SQL
df.createOrReplaceTempView('performance_reviews')

sql_result = spark.sql("""
    SELECT 
        emp_id,
        AVG(rating) AS avg_rating
    FROM performance_reviews
    GROUP BY emp_id
""")

sql_result.show()

+------+----------+
|emp_id|avg_rating|
+------+----------+
|     1|       4.5|
|     2|       4.0|
|     3|       3.0|
|     4|       5.0|
+------+----------+



In [0]:
#Window Functions
window_spec = Window.partitionBy('emp_id').orderBy('review_date')

df_with_cumulative_avg = df.withColumn(
    'cumulative_avg',
    avg('rating').over(window_spec)
)

df_with_cumulative_avg.show()

+------+-----------+-----------+------+--------+--------------+
|emp_id|review_date| department|rating|reviewer|cumulative_avg|
+------+-----------+-----------+------+--------+--------------+
|     1| 2024-01-10|Engineering|     5|    John|           5.0|
|     1| 2024-03-10|Engineering|     4|    Jane|           4.5|
|     2| 2024-01-11|         HR|     4|    Jane|           4.0|
|     2| 2024-03-11|         HR|  NULL|     Sam|           4.0|
|     3| 2024-01-12|      Sales|     3|     Sam|           3.0|
|     4| 2024-02-01|Engineering|     5|    John|           5.0|
+------+-----------+-----------+------+--------+--------------+

